In [1]:
import torch

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel

device = C.get_device()
print(f"Using device: {device}")

checkpoint_path = "../trained_models/ctc_specaugment_70epochs.pt"

checkpoint = torch.load(checkpoint_path, map_location=device)

model = CTCModel().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

Using device: cuda


CTCModel(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (14):

In [2]:
from src.ctc.features import wav_path_to_logmel
from src.ctc.dataset import textgrid_to_phone_ids
from src.ctc.metrics import greedy_decode, decode_to_phones, compute_per

wav_path = "../AutorskieDane/AutorskiDataset/id2.wav"
tg_path = "../AutorskieDane/AutorskiDataset/id2.TextGrid"
#wav_path = "../slowa_testowe/mysz.wav"
#tf_path = "../slowa_testowe/"


target_ids = textgrid_to_phone_ids(tg_path, map_sp_to_sil=True)


mel = wav_path_to_logmel(wav_path)  # (F, T)
mel = mel.unsqueeze(0).to(device)  # (1, F, T)

with torch.no_grad():
    logits = model(mel)  # (1, T', C+1)

pred_ids = greedy_decode(logits.cpu())[0]  # list[int]


per = compute_per([pred_ids], [target_ids])
print(f"PER for this utterance: {per:.4f}")

print("Target:     ", decode_to_phones(target_ids))
print("Prediction: ", decode_to_phones(pred_ids))

PER for this utterance: 0.2698
Target:      sil s t u d e n tsj i c e r u n k u sil d o v a d u j oc5 sj e sil j a k sil p o s t e m p o v a tsj sil z d u Z i2 m i i l o sj tsj a m i d a n i2 h sil p o h o dz o n c i2 m i sil z r u Z n i2 h zj r u d e w sil u tS oc5 sj e r u v n~ e S j e i n t e r p r e t o v a tsj sil f S i2 s t k o t o sil v o p a r tsj u o sil p r o g r a m i2 sil i sil a l g o r i2 t m i2 k o m p u t e r o v e sil a sil t a k S e sil v e dz e sil s m a t e m a t i2 c i sil s t a t i2 s t i2 c i sil i e k o n o m j i sil
Prediction:  sil s t u d e tsj k i e r u n k u sil t o u v i e d u j oc5 sj i e sil i j a k sil o s t eo5 p o v a tsj sil z d u Z i2 m i e i j o sj tsj a m i d o n e h sil p o v o dz oc5 c i2 m i sil z r u Z n e h sj r u d e w sil p u tS oc5 tsj i e r u v n i e S sil j e i n t e r tS r e t o v a tsj sil p S i2 s t k o t o sil o b a r tsj i w k o r o g r a m e sil i sil a l g r o t sil e k o m u t e r o v a sil a n t a h Z e v i a dz a oc5 s m a t e 